In [1]:
%pip install numpy
%pip install --upgrade matplotlib scipy
%conda install numpy=1.25 matplotlib scipy

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.

CondaValueError: You have chosen a non-default solver backend (libmamba) but it was not recognized. Choose one of: classic


Note: you may need to restart the kernel to use updated packages.


Shor's factoring algorithm


## Problem outline

We will use the quantum computer simulator we set up on problem set 2 to implement a classical simulation of Shor's factoring algorithm. The learning goal is to understand the different parts of the factoring algorithm in detail. 

We will focus on the actual quantum part. The exercises on the number theoretic aspects are optional but they can be helpful for understanding what happens in the factoring algorithm.

Note that the developing the entire simulation on your own would be quite time consuming, so we provide some routines for you and made the last part of putting together the entire factoring algorithm optional.

There is a lot of literature on educational material on this topic. For example, what was formerly known as the quiskit textbook can be found here: https://github.com/Qiskit/textbook/tree/main/notebooks/ch-algorithms# This is a collection of tutorial style jupyter notebooks, which also includes one on Shor's argorithm. 

In [6]:
# load some useful modules

# standard numerics and linear algebra libraries
import numpy as np  
import numpy.linalg as LA
import scipy.linalg as sciLA

# for making plots
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# measure runtimes
import time as time 

# sparse matrix functions
import scipy.sparse as sparse

# avoid typing np.XY all the time
from numpy import (array, pi, cos, sin, ones, size, sqrt, real, mod, append, arange, exp)

import math
from fractions import Fraction

%matplotlib inline

### RSA

As a warm up we implement the RSA encryption scheme. 
Write functions that, given two prime numbers $p$ and $q$ and a number $e<\phi(pq)$, generate a public and a private key.
Note that often, Carmichael's function $\lambda(pq)$ is used in instead of Euler's function. $\lambda(N)$ is defined as the smallest number $r$ for which $a^r=1\,\mathrm{mod}N)$ for all $a$ coprime to $N$. Show that the RSA scheme also works in this case.
Also write functions that use these key to encrypt and decrypt a message $m$.
Here $m$ can be a number $<pq$. To encryt an actual text you would first have to split the squence of letters into blocks that can be mapped to numbers and then encrypted blockwise. If you want to have some fun, write functions that encrypt and decrypt text letter wise, and use it to decrypt the word SHOR.

Hints: Useful functions `pow(a,b,N)` (i.e., $a^b\,\mathrm{mod}N$), `math.gcd()`.

As a testing example you may want to generate public and private keys with $p=61$, $q=53$, $e=17$ and encrypt and decrypt the message $m=65$. This is the example given on the wikipedia page on the RSA cryptosystem. Try out examples with N being a products of larger primes.  

If a third party can determine the prime factors of $N$, they can efficiently calculate the private key $d$ and thus decrypt the message. The security of the protocol thus relies on factoring being a hard problem classically. Thus, being able to factor $N$ efficiently with a quantum computer would break the RSA scheme.

### Success of the factoring algoithm

In the factoring algorithm, given a number $N$ of which we want to find a factor, one randomly picks a number $1<x<N$ that is coprime with $N$ and determines its order mod $N$. (In fact, we randomly pick a number $1<x<N$ and check if it is coprime wih N. If not, we have aready found a factor of N and can stop.) If the order $r$ is even and the root of $x^{r/2}\neq -1$, then either $x^{r/2}+1$ or $x^{r/2}-1$ must have a common factor with $N$. Hence by calculating $\mathrm{gcd}(x^{r/2}+1,N)$ and $\mathrm{gcd}(x^{r/2}-1,N)$ we find a factor of $N$ and the algorithm succeeds. Here we want to explore for how many randomly chosen $x$ these conditions are fulfilled, thereby learning something about the structure of the multiplicative group of coprimes with $N$. 

- Write a function that finds the order of $x$ mod $N$ in a brute force way (just iteratively multiplying $x$ with itself mod $N$ until one gets 1). This part will in the end be done on the quantum computer.
- Check for how many of the numbers coprime with $N$ does $x^{r/2}\pm 1\, \mathrm{mod}\, N$ have a common factor with $N$, i.e. the factoring algorithm succeeds. Use small $N$ which are a product of two primes for checking, e.g. $N=91$.

### The quantum Fourier transform

We now come to the quantum part of the algorithm. The first crucial ingredient of the quantum order finding algorithm is the inverse quantum Fourier transform. 

Below we provide some functions you might have developed in similar form on problem set 2, and which might be useful in the following.

In [3]:
# Collection of useful stuff from last problem set

# helper function for generating basis vectors
def basisvec(n, k):
    v = np.zeros(2**n)
    v[k] = 1
    return v

# helper function for initializing all qubits in state zero
def initRegister(n):
    return basisvec(n,0)

# helper functions to convert integer number to list of binary digits
def indToState(n, k):
    num = bin(k)[2:].zfill(n)
    return array([int(x) for x in str(num)])

# helper functions to convert list of binary digits to integer number
def stateToInd(state):
    return int("".join(str(x) for x in state),2)

def systemSizeFromState(psi):
    return int(np.log2(len(psi)))

def doMeasurement(psi):
    n = systemSizeFromState(psi)
    pvec = np.abs(psi)**2
    thresholds = np.cumsum(pvec)
    r = np.random.rand()
    indOutcome = np.sum(thresholds < r)
    return indToState(n, indOutcome)
    

def rotation(ax,theta):
    return sciLA.expm(-1j * theta/2 * (ax[0]*X + ax[1]*Y + ax[2]*Z))

def Rk(k):
    return array([[1, 0],
                  [0, exp(1j*2*pi/2**k)]])

def buildSparseGateSingle(n, i, gate):
    sgate = sparse.csr_matrix(gate)
    return sparse.kron(sparse.kron(sparse.identity(2**i), sgate), sparse.identity(2**(n-i-1)))

P0 = array([[1,0],
            [0,0]])
P1 = array([[0,0],
            [0,1]])
C01 = array([[0,1],
             [0,0]])
C10 = array([[0,0],
             [1,0]])

def buildSparseCNOT(n, ic, it):
    P0ic = buildSparseGateSingle(n, ic, P0)
    P1ic = buildSparseGateSingle(n, ic, P1)
    Xit  = buildSparseGateSingle(n, it, X)
    return P0ic + P1ic @ Xit

#### (a) Building swap gates and controlled phase gates

- Write a function that builds controlled $R_k$ gates between two arbitrary quibits in a register (using sparse matrices), similar to the CNOT gate you implemented on problem set 2. The single qubit $R_k$ gate is provided above but keep in mind that for the inverse QFT you need $R_k^{-1}=R_k^\dagger$.
- Write a function that builds swap gates between pairs of qubits. Note that a swap gate on two qubits can be written as $U_{\textrm{swap}} = |00\rangle\langle 00|+|01\rangle\langle 10|+|10\rangle\langle 01|+|11\rangle\langle 11|$.

#### (b) Fourier transform

Implement the inverse quantum Fourier transform (iQFT). 

Instructions:

Write a function that takes as input the state of the register (i.e. the coefficient vector in the computational basis), sequentially applies all gates of the iQFT circuit, and outputs the final state (coefficient vector) again.
We don't want to build the whole iQFT as a unitary matrix as this matrix would be dense and take up a lot of memory. Applying it to a state would scale as $(2^n)^2$. The sequential application of the $O(n^2)$ sparse gates results in a $O(n^2 2^n)$ scaling, thus you have implemented a version of a classical (inverse) fast Fourier transform!

For the order finding algorithm we will need to apply the iQFT only to the first register. 
Thus, your iQFT function should allow you to apply it only to the first $n_1$ qubits of an $n$-qubit register and leave the remaining qubits alone, i.e. is the identity on the remaining qubits.

Note: In the order finding algorithm, the qubits will be measured right after the iQFT. Thus, as shown on Problem set 3.1, it would be sufficent to measure the qubits sequentially and feed forward the result. This would enable to simulate this part of the order finding algorithm more efficiently, and only using single qubit gates.

### Quantum order finding

The second ingredient of the order finding algorithm is the modular multiplication gate: $U:\,|y\rangle \mapsto |xy \,\mathrm{mod}\, N\rangle$ (for $y< N$) in second register controlled by some qubit in the first register. For $y\geq N$, $U$ is the identity. This is the "padding" we need since $N$ will not be a power of two. $U^{2^j}$ gates are applied to the second register controlled by the state of qubit $j$ in the first register. When doing this on an actual quantum computer this step would be accomlished by implementing modular exponentiation with reversible classical gates. This requires $O(L^3)$ gates (where $L$ is the number of binary digits of $N$) classically through repeated squaring. 
This is actually the bottleneck of the factoring algorithm and constructing the classical reversible circuits is not trivial.

Here we will make our life easy and directly implement the collective action of the controlled $U^{2^j}$ gates, which is the unitary $U_x :|z\rangle|y\rangle \mapsto |z\rangle|x^zy \,\mathrm{mod}\, N\rangle$ for $y< N$ and $|z\rangle|y\rangle \mapsto |z\rangle|y\rangle$ otherwise. 

#### (a) Modular exponentiation gate

Below you are provided a function that builds the operation $U_x$ as a sparse matrix. The inputs are the number of qubits $t$ in the first register and the integer numbers $x$ and $N$. The resulting matrix only has one non-zero entry in each row. Let us explain more precisely what the structure of the matrix $U_x$ is: It is block diagonal. For a fixed state $|z\rangle$ of the 1st register, there is a $2^L \times 2^L$ block ($L$ being the size of the second register) spanned by the all possible states $| y \rangle$ of the second register. Within each such block in each row there is one non-zero matrix element (with value 1) at position (column) $x^zy \,\mathrm{mod}\, N$ (for $y<N$) with respect to the beginning of the block. For $y\geq N$ the non-zero entry is on the diagonal. 

Make sure you understand how the construction of the $U_x$ gate works and test it for some examples where you can compute the positions of the non-zero matrix elements by hand to test the implemenation.

In [4]:
def qubits_for_number(N):
    "Return minimum number of qubits needed to encode L"
    return int(np.ceil(np.log2(N)))

def build_x_tothe_z(t, x, N):
    L = qubits_for_number(N)
    dim1=2**t
    dim2=2**L
    dim=dim1*dim2
    row=arange(dim) # indexes all rows
    col=arange(dim) # will contain the position (col) of the non-zero entry for each row
    for j in range(dim1): # loop over states of the first register
        for y in range(dim2): # loop over states of the second register
            if y < N: # if y>=N, we want the identity, so we leave the column index unchanged (it was initialized to be equal to the row index)
                col[j*dim2+y]=j*dim2 + np.mod(y*pow(x,j,N),N)
    return sparse.csr_matrix((np.ones(dim), (row, col)))

# testing
print(build_x_tothe_z(2,3,3))

  (0, 0)	1.0
  (1, 1)	1.0
  (2, 2)	1.0
  (3, 3)	1.0
  (4, 4)	1.0
  (5, 4)	1.0
  (6, 4)	1.0
  (7, 7)	1.0
  (8, 8)	1.0
  (9, 8)	1.0
  (10, 8)	1.0
  (11, 11)	1.0
  (12, 12)	1.0
  (13, 12)	1.0
  (14, 12)	1.0
  (15, 15)	1.0


#### (b) The order finding algorithm put together

Now implement the quantum order finding algorithm (excluding the final measurement) and use it to find the order of 7 mod 15 and of 2 mod 21. Inputs: t, x, N. Output: The final state of the circuit. 

Analyse the output state: In what states can each one of the registers end up? Visualize the probability distribution over the outcome states. Interpret and discuss your results! The code snippets below give you some ideas on how to do this. 

Try out different $t$ to see how many qubits are needed in the 1st register to get reliable results. Try to verify the expected behavior: In the case $N=15$ the order finding algorithm always gives the exact result because the order is a power of 2. For $N=21$ this is not always the case (what orders $r$ can appear?), meaning that the peaks in the probability distribution over the predicted phases becomes broader when $t$ is decreased.

When the denominator of $s/r$ is not a power of two, the phase can only be estimated approximately. In this case the continued fraction algorithm is used to find the the closest rational with denominator $<N$ from the output $b/2^t$ of the order finding algorithm. Check for which outputs $b$ the correct order $r$ is found. 

The continued fraction algorithm is in principle straight forward to implement (we provide an example at the end of this problem set) but you can as well use the function Fraction() from the fractions module, as done in the snippet below.

In [5]:
# Here are some ideas how to analyze the probability distribution, assuming psi holds the oputput state of the order finding algorithm

# analyse the probabilities
Probs=np.abs(psi**2)

# plot the distribution
plt.semilogy(Probs,'.')

# determine where the peaks are (round small numbers down to zero)
indNZ=np.nonzero(np.round(Probs,2))

L = qubits_for_number(N)
# probable states of the first register 
indNZ_reg1= array(indNZ)//2**L
print(indNZ_reg1)
print(np.round(Probs[indNZ],4)) # values at the peaks
print(sum(Probs[indNZ])) # total of peak values (probability to obtain the closest possible number to s/r, but this is not true in general, depends on how we round)

# probable states of the second register
indNZ_reg2= array(indNZ) % 2**L
print(indNZ_reg2)


# Here is an example of how to obtain the nearest rational number with denominator <N (here N=15)
frac = Fraction(33/256).limit_denominator(15)
print(frac.numerator)
print(frac.denominator)

NameError: name 'psi' is not defined

### Factoring algorithm

Now we want to put the whole factoring algorithm together, including measurements.

The functions and pseudocode below will help you with this.

Use your algorithm to factor 21. Play around and try larger numbers and vary the size of the 1st register. How far can you get in terms of the size of N?

Hint: To speed up you simulation you can "cheat" and run the order finding algorithm only once for a given $x$ and use the resulting probability distribution to repeatedly do measurements without re-evaluating the circuit.

In [ ]:
def systemSizeFromState(psi):
    return int(np.log2(len(psi)))

def doMeasurement(pvec):
    n = systemSizeFromState(pvec)
    thresholds = np.cumsum(pvec)
    r = np.random.rand()
    indOutcome = np.sum(thresholds < r)
    return indToState(n, indOutcome)
    

```
factoring(inputs):
    determine the size of the second register
    while no factor was found try different values of x
        check if x is coprime with N (if not, then we already have a factor)
        run the order finding algorithm, including the final measurement
        using the result in the first register, use continued fraction to get a guess for r
        while r is not yet the order (and some maximal number of trials has not been exceeded)
            run the order finding algorithm again to get a second estimate r2 for the order
            r=lcm(r, r2)
        if the conditions r even and x^(r/2)!=-1 mod N are fulfilled
            return gcd(x^(r/2) + 1, N) or gcd(x^(r/2) + 1, N) Success!
```

### Continued fraction algorithm

Here is a "manual" implementation of the continued fraction expansion, i.e. a function that takes as input a real number phi and returns a rational approximation of it. The function takes an additional integer input that specifies the maximal value of the denominator of the approximating fraction (i.e. the requested precision). Playing with this may help you to deepen your understanding of continued fractions.

In [ ]:
# Continued fraction (see wikipedia)

def cont_frac(phi,max_denom):

    frac=[]

    #integer part
    a=phi//1
    r=phi-a
    frac.append(int(a))

    while eval_contfrac_rational(frac)[1]<=max_denom and r>1/max_denom:
        a=(1/r)//1
        r=(1/r)-a
        frac.append(int(a))
    if r<1/max_denom:
        return(frac)
    else:
        return(frac[:-1])

def eval_contfrac(frac):
    n=len(frac)
    a=0
    for i in range(n-1,0,-1):
        a=1/(frac[i]+a)
    a=frac[0]+a
    return a

def eval_contfrac_rational(frac):
    n=len(frac)
    if n==1:
        return [frac[0],1]
    numer=1
    denom=frac[n-1]
    for i in range(n-2,0,-1):
        denom_new=denom*frac[i] + numer
        numer=denom
        denom=denom_new
    numer = frac[0]*denom + numer
    return [numer, denom]

# testing:
# a rational number
phi=251/2048
print(phi)
max_denom=200

frac = cont_frac(phi,max_denom)
print(frac)
print(eval_contfrac(frac))
print(eval_contfrac_rational(frac))

frac2 = Fraction(phi).limit_denominator(max_denom)
print(frac2.numerator)
print(frac2.denominator)

### Man vs. machine

Ask Chat GPT to write a python program that emulates Shor's algorithm (I haven't tried). See if what comes out is correct, and if so, check whether your emulator beats it in terms of runtime.